# 03 - Bronze Layer: Live Telematics Streaming (Kafka)

**Project:** Auto Insurance Claims & Telematics Analytics

## What this notebook does
Connects directly to a live Kafka topic on Confluent Cloud and streams
telematics events into a bronze Delta table in near-real time, using genuine
continuous Structured Streaming - not the batch-style
trigger(availableNow=True) pattern used for file-based sources in the
Healthcare and P&C projects.

## Important
Once started, this streaming query runs indefinitely (or until manually
stopped) - it does not "finish" the way a batch cell does. This is expected
behavior for a true streaming source with no natural end, unlike a folder of
files that eventually has nothing left to process.

## Table created
- `main.auto_insurance_telematics.bronze_telematics_events`

In [0]:
bootstrap_server = dbutils.secrets.get(scope="kafka", key="bootstrap-server")
api_key = dbutils.secrets.get(scope="kafka", key="api-key")
api_secret = dbutils.secrets.get(scope="kafka", key="api-secret")

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
        f'required username="{api_key}" password="{api_secret}";'
    ),
    "subscribe": "telematics_events",
    "startingOffsets": "earliest",
    "failOnDataLoss": "false",
}

raw_kafka_stream = spark.readStream.format("kafka").options(**kafka_options).load()

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("speed_mph", DoubleType(), True),
    StructField("event_timestamp", StringType(), True),
])

bronze_telematics_stream = (
    raw_kafka_stream
    .withColumn("value_str", col("value").cast("string"))
    .withColumn("parsed", from_json(col("value_str"), event_schema))
    .select(
        col("parsed.event_id").alias("event_id"),
        col("parsed.vehicle_id").alias("vehicle_id"),
        col("parsed.event_type").alias("event_type"),
        col("parsed.speed_mph").alias("speed_mph"),
        col("parsed.event_timestamp").alias("event_timestamp"),
        col("key").cast("string").alias("kafka_key"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp"),
        current_timestamp().alias("_ingested_at"),
    )
)

In [0]:
checkpoint_path = "/Volumes/main/auto_insurance_telematics/checkpoints/bronze_telematics_events"

(
    bronze_telematics_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("main.auto_insurance_telematics.bronze_telematics_events")
)

In [0]:
%sql
SELECT COUNT(*) FROM main.auto_insurance_telematics.bronze_telematics_events;


COUNT(*)
156


In [0]:
%sql
SELECT * FROM main.auto_insurance_telematics.bronze_telematics_events LIMIT 10;

event_id,vehicle_id,event_type,speed_mph,event_timestamp,kafka_key,kafka_partition,kafka_offset,kafka_timestamp,_ingested_at
70e2c9a8-ad8a-48ca-81ef-6cc0951ad728,VEH0025,normal_driving,37.1,2026-08-27T13:43:23.580966+00:00,VEH0025,5,0,2026-08-27T13:43:23.580Z,2026-08-29T00:26:32.647Z
28649db6-a279-4331-b0b5-62049d57805c,VEH0004,rapid_acceleration,42.9,2026-08-27T13:43:26.206582+00:00,VEH0004,5,1,2026-08-27T13:43:26.206Z,2026-08-29T00:26:32.647Z
4166c164-f43d-4e7c-8a4b-8873a063cacc,VEH0005,normal_driving,20.4,2026-08-27T13:43:30.985611+00:00,VEH0005,5,2,2026-08-27T13:43:30.985Z,2026-08-29T00:26:32.647Z
d9cf91f3-6816-4466-b246-c26c3c3090cd,VEH0018,hard_brake,38.7,2026-08-27T13:43:36.274768+00:00,VEH0018,5,3,2026-08-27T13:43:36.274Z,2026-08-29T00:26:32.647Z
414d7df5-3eb4-4a98-8c29-01130e67436d,VEH0007,normal_driving,62.8,2026-08-27T13:43:56.292930+00:00,VEH0007,5,4,2026-08-27T13:43:56.292Z,2026-08-29T00:26:32.647Z
0126fb67-00eb-4bd2-92a5-b0fc6e32a02e,VEH0018,normal_driving,29.7,2026-08-27T13:43:59.291073+00:00,VEH0018,5,5,2026-08-27T13:43:59.290Z,2026-08-29T00:26:32.647Z
0608d213-d67c-4de3-8ef2-3d2f0cb73078,VEH0004,speeding,75.7,2026-08-27T13:44:10.951433+00:00,VEH0004,5,6,2026-08-27T13:44:10.951Z,2026-08-29T00:26:32.647Z
89eab47a-7bb2-47a3-8318-f30b77ce1c8e,VEH0018,normal_driving,38.3,2026-08-27T13:44:11.899704+00:00,VEH0018,5,7,2026-08-27T13:44:11.899Z,2026-08-29T00:26:32.647Z
1ce1dc55-6841-498f-97af-ff830b5e9048,VEH0005,normal_driving,52.5,2026-08-27T13:44:13.871572+00:00,VEH0005,5,8,2026-08-27T13:44:13.871Z,2026-08-29T00:26:32.647Z
80a17246-2090-494c-b519-973b6ab2b28b,VEH0007,rapid_acceleration,54.1,2026-08-27T13:44:14.400136+00:00,VEH0007,5,9,2026-08-27T13:44:14.399Z,2026-08-29T00:26:32.647Z


In [0]:
%sql
SELECT kafka_partition, COUNT(*) AS row_count
FROM main.auto_insurance_telematics.bronze_telematics_events
GROUP BY kafka_partition
ORDER BY kafka_partition;

kafka_partition,row_count
0,26
1,21
2,23
3,19
4,39
5,28
